In [10]:
import asyncio
import nest_asyncio
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from mcp.types import TextContent
import subprocess

nest_asyncio.apply()  # Needed to run interactive python
print("Imports loaded. nest_asyncio applied — async code can now run in Jupyter.")

Imports loaded. nest_asyncio applied — async code can now run in Jupyter.


In [11]:
server_params = StdioServerParameters(
    command="python",
    args=["server.py"],
)

In [12]:
async def explore_tools():
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            print("stdio_client opened — server subprocess is now running.")
            print("ClientSession created — MCP protocol handshake in progress...")

            await session.initialize()
            print("Session initialized! Client and server have negotiated capabilities.\n")

            tools_result = await session.list_tools()
            print("Available tools advertised by the server:")
            for tool in tools_result.tools:
                print(f"   - {tool.name}: {tool.description}")

asyncio.run(explore_tools())


stdio_client opened — server subprocess is now running.
ClientSession created — MCP protocol handshake in progress...
Session initialized! Client and server have negotiated capabilities.

Available tools advertised by the server:
   - add: Add two numbers together


In [14]:
async def call_add_tool():
    async with stdio_client(server_params, errlog=subprocess.DEVNULL) as (read_stream, write_stream):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            print("Calling tool: 'add' with arguments a=2, b=3")

            result = await session.call_tool("add", arguments={"a": 2, "b": 3})
            content = result.content[0]

            if isinstance(content, TextContent):
                print(f"Raw response content type : TextContent")
                print(f"Result: 2 + 3 = {content.text}")

asyncio.run(call_add_tool())


Calling tool: 'add' with arguments a=2, b=3
Raw response content type : TextContent
Result: 2 + 3 = 5
